# Steam Oyun Benzerlik Ağı – Etkileşimli Ağ
Beril Özdizdar · Sosyal Ağ Analizi dönem ödevi

Bu defter tek bir **`index.html`** üretir: arama, oyuna tıklayınca detay paneli, topluluk filtresi ve TR/EN dil seçeneği olan etkileşimli ağ.
Sadece `steam-network.zip` gerekir. Hücreleri sırayla çalıştır, zip'i seç. Sonunda `sonuclar4.zip` iner, içindeki `index.html`'i tarayıcıda açıp dene.

## 0. Kurulum ve veri

In [ ]:
!pip -q install igraph leidenalg

In [ ]:
import os, glob, json, subprocess, warnings
import numpy as np, pandas as pd, networkx as nx
from google.colab import files
warnings.filterwarnings("ignore")

ROOT = "/content"
def bul(desen):
    return glob.glob(f"{ROOT}/**/{desen}", recursive=True)

if not bul("processed/edges_game_game.csv"):
    if not glob.glob(f"{ROOT}/steam-network*.zip"):
        print("Açılan pencereden steam-network.zip dosyasını seç:")
        files.upload()
    for z in glob.glob(f"{ROOT}/steam-network*.zip"):
        subprocess.run(["unzip", "-q", "-o", z, "-d", ROOT])

assert bul("processed/edges_game_game.csv"), "steam-network.zip yüklenmeli."
OUT4 = f"{ROOT}/sonuclar4"; os.makedirs(OUT4, exist_ok=True)
print("Veri bulundu.")

## 1. Toplulukları hesapla (1. defterdeki yöntemle)

In [ ]:
import igraph as ig, leidenalg

gt = pd.read_csv(bul("processed/edges_game_tag.csv")[0], encoding="utf-8-sig")
gg = pd.read_csv(bul("processed/edges_game_game.csv")[0], encoding="utf-8-sig")
meta = {}
for line in open(bul("raw/appdetails.jsonl")[0], encoding="utf-8"):
    if line.strip():
        d = json.loads(line)
        meta[d["appid"]] = (d.get("name", ""), d.get("positive", 0) + d.get("negative", 0))

G = nx.Graph()
G.add_nodes_from(gt["appid"].unique())
G.add_weighted_edges_from(gg[["appid_a", "appid_b", "benzerlik"]].itertuples(index=False, name=None))
nodes = list(G.nodes()); idx = {n: i for i, n in enumerate(nodes)}
g_ig = ig.Graph(n=len(nodes), edges=[(idx[u], idx[v]) for u, v in G.edges()])
g_ig.es["weight"] = [d["weight"] for _, _, d in G.edges(data=True)]
en_iyi = None
for sd in range(10):
    part = leidenalg.find_partition(g_ig, leidenalg.ModularityVertexPartition,
                                    weights="weight", n_iterations=-1, seed=sd)
    if en_iyi is None or part.modularity > en_iyi.modularity:
        en_iyi = part
com = pd.DataFrame({"appid": nodes, "topluluk": en_iyi.membership,
                    "oyun": [meta.get(n, ("", 0))[0] for n in nodes],
                    "degerlendirme": [meta.get(n, ("", 0))[1] for n in nodes]})
print(f"Leiden: modülerlik {en_iyi.modularity:.4f}, {com['topluluk'].nunique()} topluluk")

## 2. Gösterilecek ağı seç ve yerleştir

In [ ]:
N_GOSTER = 400      # sayfada gösterilecek en çok değerlendirilen oyun sayısı
ESIK = 0.65         # bu benzerliğin altındaki bağlantılar çizilmez

sizes = com["topluluk"].value_counts()
top5 = list(sizes.index[:5])
sel = com.nlargest(N_GOSTER, "degerlendirme")
selset = set(sel["appid"])
e = gg[(gg["benzerlik"] >= ESIK) & gg["appid_a"].isin(selset) & gg["appid_b"].isin(selset)]
H = nx.Graph()
H.add_weighted_edges_from(e[["appid_a", "appid_b", "benzerlik"]].itertuples(index=False, name=None))
info = com.set_index("appid")

# topluluk merkezli yerleşim (posterdeki gibi)
merkez = {c: (1.3 * np.cos(2 * np.pi * i / 5 + np.pi / 2), np.sin(2 * np.pi * i / 5 + np.pi / 2)) for i, c in enumerate(top5)}
merkez[-1] = (0.0, 0.0)
grp = {}
for n in H.nodes():
    c = info.loc[n, "topluluk"]
    grp.setdefault(c if c in merkez else -1, []).append(n)
pos = {}
for c, ns in grp.items():
    sub = H.subgraph(ns)
    p = nx.spring_layout(sub, weight="weight", seed=42, iterations=200) if len(ns) > 1 else {ns[0]: np.zeros(2)}
    ext = np.abs(np.array(list(p.values()))).max() or 1
    r = min(0.42, 0.07 * np.sqrt(len(ns)) + 0.1)
    for n, v in p.items():
        pos[n] = (merkez[c][0] + r * v[0] / ext, merkez[c][1] + r * v[1] / ext)
print(f"Gösterilen: {H.number_of_nodes()} oyun, {H.number_of_edges()} bağlantı")

## 3. index.html üret

In [ ]:
HTML_SABLON = r'''<!DOCTYPE html>
<html lang="tr"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Steam Oyun Benzerlik Ağı</title>
<style>
:root{--bg:#0f141b;--panel:#172231;--card:#1c2a3b;--fg:#e8eaed;--mut:#9aa4b2;--acc:#66c0f4}
*{box-sizing:border-box}
html,body{margin:0;height:100%}
body{display:flex;flex-direction:column;height:100vh;background:var(--bg);color:var(--fg);font-family:Inter,Arial,sans-serif}
#top{display:flex;gap:16px;align-items:center;padding:12px 20px;background:var(--panel);border-bottom:1px solid #2b4258;flex-wrap:wrap}
#top h1{font-size:20px;margin:0;color:#f2f4f7}
#sw{position:relative;flex:1;min-width:200px;max-width:420px}
#q{width:100%;padding:10px 14px;border-radius:10px;border:1px solid #2b4258;background:var(--bg);color:var(--fg);font-size:15px}
#sug{display:none;position:absolute;left:0;right:0;top:44px;background:var(--card);border:1px solid #2b4258;border-radius:10px;z-index:20;overflow:hidden}
#sug button{display:block;width:100%;text-align:left;padding:9px 14px;background:none;border:0;color:var(--fg);font-size:14px;cursor:pointer}
#sug button:hover{background:#243649}
#lang{padding:8px 14px;border-radius:10px;border:1px solid #2b4258;background:var(--card);color:var(--fg);cursor:pointer;font-size:14px}
#wrap{flex:1;min-height:0;display:flex}
#side{width:270px;padding:16px;background:var(--panel);overflow:auto;border-right:1px solid #2b4258}
#cy{flex:1;min-width:0}
#info{width:320px;padding:16px;background:var(--panel);overflow:auto;border-left:1px solid #2b4258}
h2{font-size:22px;margin:0 0 8px;color:#f2f4f7}
h3{font-size:15px;margin:18px 0 8px;color:var(--acc)}
.lg{display:flex;gap:10px;align-items:flex-start;margin:10px 0;font-size:14px;line-height:1.3;cursor:pointer}
.lg em,.chip em{color:var(--mut);font-style:normal}
.dot{flex:none;width:14px;height:14px;border-radius:50%;margin-top:2px}
.note,.hint{color:var(--mut);font-size:13px;line-height:1.45}
.pill{display:inline-block;margin:0 0 10px;padding:3px 12px;border:1px solid;border-radius:20px;font-size:13px}
.chip{display:inline-block;margin:0 6px 6px 0;padding:4px 10px;background:var(--card);border-radius:14px;font-size:13px}
ul{margin:0;padding-left:18px;font-size:14px;line-height:1.7}
a{color:var(--acc);text-decoration:none}
@media (max-width:900px){#wrap{flex-direction:column}#side,#info{width:100%;max-height:28%}#cy{min-height:45%}}
</style></head>
<body>
<div id="top"><h1 id="ttl"></h1><div id="sw"><input id="q" type="search" autocomplete="off"><div id="sug"></div></div><button id="lang"></button></div>
<div id="wrap"><div id="side"></div><div id="cy"></div><div id="info"></div></div>
<script src="https://cdn.jsdelivr.net/npm/cytoscape@3.30.2/dist/cytoscape.min.js"></script>
<script>
(function(){
const D = __DATA__;
const COL = ["#66c0f4","#a3cf06","#f5a524","#e0609b","#b18cff","#5b6676"];
const NAMES = {
 tr:["Aksiyon-RPG, açık dünya","Strateji, simülasyon","FPS, çok oyunculu nişancı","Korku, atmosferik macera","Bağımsız, 2D, parti oyunları","Diğer küçük gruplar"],
 en:["Action RPG, open world","Strategy, simulation","FPS, multiplayer shooters","Horror, atmospheric adventure","Indie, 2D, party games","Other small groups"]
};
const TXT = {
 tr:{title:"Steam Oyun Benzerlik Ağı",ph:"Oyun ara…",legend:"Topluluklar",rev:"Değerlendirme",tags:"En çok oy alan etiketler",sim:"Bağlı oyunlar (en çok değerlendirilenler)",none:"Bağlı oyun yok",lang:"English",
  hint:"Bir oyuna tıkla ya da yukarıdan ara. Renk topluluğu, büyüklük değerlendirme sayısını gösterir. Sürükleyerek gez, tekerlekle yakınlaştır.",
  note:function(m){return "En çok değerlendirilen "+m.n+" oyun gösteriliyor. Etiket benzerliği "+m.th+" ve üzeri olan "+m.e+" bağlantı çizildi.";}},
 en:{title:"Steam Game Similarity Network",ph:"Search a game…",legend:"Communities",rev:"Reviews",tags:"Top-voted tags",sim:"Connected games (most reviewed)",none:"No connected games",lang:"Türkçe",
  hint:"Click a game or search above. Colour shows the community, size shows the number of reviews. Drag to pan, scroll to zoom.",
  note:function(m){return "Showing the "+m.n+" most-reviewed games. "+m.e+" links with tag similarity of "+m.th+" or higher are drawn.";}}
};
let lang = "tr";
const vis = [true,true,true,true,true,true];
const lrs = D.nodes.map(function(n){return Math.log10(n.r+1);});
const lo = Math.min.apply(null,lrs), hi = Math.max.apply(null,lrs);
const els = [];
D.nodes.forEach(function(n,i){els.push({data:{id:"n"+i,label:n.l,c:n.c,lr:lrs[i],i:i},position:{x:n.x,y:n.y}});});
D.edges.forEach(function(e,k){els.push({data:{id:"e"+k,source:"n"+e[0],target:"n"+e[1]}});});
const cy = cytoscape({container:document.getElementById("cy"),elements:els,layout:{name:"preset"},
 minZoom:0.15,maxZoom:8,wheelSensitivity:0.25,
 style:[
  {selector:"node",style:{"background-color":function(e){return COL[e.data("c")];},"width":"mapData(lr,"+lo+","+hi+",8,40)","height":"mapData(lr,"+lo+","+hi+",8,40)","border-width":1,"border-color":"#0f141b"}},
  {selector:"node.top, node.sel",style:{"label":"data(label)","color":"#f2f4f7","font-size":13,"font-weight":700,"text-valign":"top","text-margin-y":-4,"text-outline-color":"#0f141b","text-outline-width":3}},
  {selector:"edge",style:{"width":0.6,"line-color":"#6f8fb0","opacity":0.14,"curve-style":"haystack"}},
  {selector:".dim",style:{"opacity":0.08}},
  {selector:"edge.hl",style:{"opacity":0.8,"width":1.6,"line-color":"#f2f4f7"}},
  {selector:"node.sel",style:{"border-width":4,"border-color":"#ffffff"}}
 ]});
lrs.map(function(v,i){return [v,i];}).sort(function(a,b){return b[0]-a[0];}).slice(0,18).forEach(function(p){cy.getElementById("n"+p[1]).addClass("top");});
cy.fit(undefined,30);

const side=document.getElementById("side"), info=document.getElementById("info");
const q=document.getElementById("q"), sug=document.getElementById("sug"), btn=document.getElementById("lang");
let cur=null;
function esc(s){return String(s).replace(/[&<>"']/g,function(m){return {"&":"&amp;","<":"&lt;",">":"&gt;",'"':"&quot;","'":"&#39;"}[m];});}
function norm(s){return s.toLocaleLowerCase("tr").normalize("NFD").replace(/[\u0300-\u036f]/g,"");}
const NN=D.nodes.map(function(n){return norm(n.l);});

function renderLegend(){
 const t=TXT[lang]; let h="<h3>"+t.legend+"</h3>";
 for(let c=0;c<6;c++){ if(!D.counts[c]) continue;
  h+='<label class="lg"><input type="checkbox" data-c="'+c+'"'+(vis[c]?" checked":"")+'><span class="dot" style="background:'+COL[c]+'"></span><span>'+NAMES[lang][c]+" <em>("+D.counts[c]+")</em></span></label>";
 }
 side.innerHTML=h+'<p class="note">'+t.note(D.meta)+"</p>";
}
function applyVis(){cy.batch(function(){cy.nodes().forEach(function(n){n.style("display",vis[n.data("c")]?"element":"none");});});}
function hint(){info.innerHTML='<p class="hint">'+TXT[lang].hint+"</p>";}
function showInfo(i){
 const t=TXT[lang], n=D.nodes[i]; cur=i;
 const nb=cy.getElementById("n"+i).neighborhood("node").map(function(x){return x.data("i");}).sort(function(a,b){return D.nodes[b].r-D.nodes[a].r;}).slice(0,8);
 let h="<h2>"+esc(n.l)+'</h2><p class="pill" style="border-color:'+COL[n.c]+";color:"+COL[n.c]+'">'+NAMES[lang][n.c]+"</p>";
 h+="<p><b>"+t.rev+":</b> "+n.r.toLocaleString(lang==="tr"?"tr-TR":"en-US")+"</p>";
 h+="<h3>"+t.tags+"</h3><div>"+n.t.map(function(x){return '<span class="chip">'+esc(x[0])+" <em>"+x[1]+"</em></span>";}).join("")+"</div>";
 h+="<h3>"+t.sim+"</h3>"+(nb.length?"<ul>"+nb.map(function(j){return '<li><a href="#" data-i="'+j+'">'+esc(D.nodes[j].l)+"</a></li>";}).join("")+"</ul>":"<p>"+t.none+"</p>");
 info.innerHTML=h;
}
function clearSel(){cy.elements().removeClass("dim sel hl");cur=null;hint();}
function select(i){
 const node=cy.getElementById("n"+i);
 cy.elements().removeClass("dim sel hl");
 cy.elements().not(node.closedNeighborhood()).addClass("dim");
 node.connectedEdges().addClass("hl"); node.addClass("sel"); showInfo(i);
}
function pick(i){
 const c=D.nodes[i].c; if(!vis[c]){vis[c]=true;applyVis();renderLegend();}
 q.value=D.nodes[i].l; sug.style.display="none"; select(i);
 cy.animate({center:{eles:cy.getElementById("n"+i)},zoom:2.2},{duration:450});
}
function applyLang(){
 const t=TXT[lang]; document.documentElement.lang=lang; document.title=t.title;
 document.getElementById("ttl").textContent=t.title; q.placeholder=t.ph; btn.textContent=t.lang;
 renderLegend(); if(cur===null){hint();}else{showInfo(cur);}
}
cy.on("tap","node",function(e){select(e.target.data("i"));});
cy.on("tap",function(e){if(e.target===cy){clearSel();}});
side.addEventListener("change",function(e){const c=e.target.getAttribute("data-c"); if(c!==null){vis[+c]=e.target.checked;applyVis();}});
info.addEventListener("click",function(e){const a=e.target.closest("a[data-i]"); if(a){e.preventDefault();pick(+a.getAttribute("data-i"));}});
q.addEventListener("input",function(){
 const v=norm(q.value.trim()); if(v.length<2){sug.style.display="none";return;}
 const m=[]; for(let i=0;i<NN.length&&m.length<8;i++){if(NN[i].indexOf(v)>=0){m.push(i);}}
 sug.innerHTML=m.map(function(i){return '<button data-i="'+i+'">'+esc(D.nodes[i].l)+"</button>";}).join("");
 sug.style.display=m.length?"block":"none";
});
sug.addEventListener("click",function(e){const b=e.target.closest("button"); if(b){pick(+b.getAttribute("data-i"));}});
q.addEventListener("keydown",function(e){if(e.key==="Escape"){q.value="";sug.style.display="none";clearSel();}});
btn.addEventListener("click",function(){lang=lang==="tr"?"en":"tr";applyLang();});
applyLang();
})();
</script></body></html>
'''

gosterilen = list(H.nodes())
sira = {n: i for i, n in enumerate(gosterilen)}
sub_gt = gt[gt["appid"].isin(set(gosterilen))].sort_values("oy", ascending=False)
etiket = {a: [[t, int(v)] for t, v in zip(g["etiket"].head(6), g["oy"].head(6))] for a, g in sub_gt.groupby("appid")}

def c_idx(c):
    return top5.index(c) if c in top5 else 5

veri_dugum = []
for n in gosterilen:
    ad, rev = meta.get(n, ("", 0))
    veri_dugum.append({"l": ad, "c": c_idx(info.loc[n, "topluluk"]), "r": int(rev),
                       "x": round(pos[n][0] * 600, 1), "y": round(-pos[n][1] * 600, 1),
                       "t": etiket.get(n, [])})
sayim = [sum(1 for d in veri_dugum if d["c"] == k) for k in range(6)]
veri = {"nodes": veri_dugum,
        "edges": [[sira[u], sira[v]] for u, v in H.edges()],
        "counts": sayim,
        "meta": {"n": len(veri_dugum), "e": H.number_of_edges(), "th": ESIK}}
json_str = json.dumps(veri, ensure_ascii=False).replace("</", "<\\/")
html = HTML_SABLON.replace("__DATA__", json_str)
with open(f"{OUT4}/index.html", "w", encoding="utf-8") as f:
    f.write(html)
print(f"index.html yazıldı: {len(html)/1024:.0f} KB")

## 4. İndir

In [ ]:
import shutil
shutil.make_archive(f"{ROOT}/sonuclar4", "zip", OUT4)
files.download(f"{ROOT}/sonuclar4.zip")